<h1 style="text-align:center; font-size:42px; margin-top:40px;">
Normalisation & correction des incohérences – Survey 2024
</h1>

## 1. Contexte et objectif

Ce module intervient après :

- la suppression des doublons (`V1`) ;
- le traitement des valeurs manquantes et des outliers (`V2`).

L’objectif est de produire une version **normalisée** du dataset, prête pour l’analyse exploratoire et la construction de modèles, en :

- corrigeant les incohérences de libellés (ex. normalisation des pays) ;
- créant des variables dérivées plus interprétables (feature engineering) ;
- décomposant certaines colonnes multivaluées ou mixtes (ex. `Employment`) ;
- sauvegardant un dataset final `V3` cohérent et exploitable.

Le fichier généré à la fin (`survey-data_V3_correct.csv`) sera utilisé dans les notebooks d’EDA et d’analyse.


## 2. Librairies et chargement des données

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../Data/Survey/processed/survey-data_V2_clean_missing_outliers.csv")
df.head()


,ResponseId,MainBranch,Age,RemoteWork,Employment,Country,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,...,SOAccount,SOPartFreq,AISelect,AIBen,AIChallenges,JobSat,Industry,OpSysProfessional use,NEWCollabToolsHaveWorkedWith,AIToolCurrently Using
0,1,I am a developer by profession,Under 18 years old,Remote,"Employed, full-time",United States of America,Hobby,Primary/elementary school,Books / Physical media,NaN,...,NaN,NaN,Yes,Increase productivity,NaN,7.0,Unknown,NaN,NaN,NaN
1,2,I am a developer by profession,35-44 years old,Remote,"Employed, full-time",United Kingdom of Great Britain and Northern I...,Hobby;Contribute to open-source projects;Other...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,Yes,Multiple times per day,"No, and I don't plan to",NaN,NaN,7.0,Unknown,MacOS,PyCharm;Visual Studio Code;WebStorm,NaN
2,3,I am a developer by profession,45-54 years old,Remote,"Employed, full-time",United Kingdom of Great Britain and Northern I...,Hobby;Contribute to open-source projects;Other...,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,Yes,Multiple times per day,"No, and I don't plan to",NaN,NaN,7.0,Unknown,Windows,Visual Studio,NaN
3,4,I am learning to code,18-24 years old,"Hybrid (some remote, some in-person)","Student, full-time",Canada,NaN,Some college/university study without earning ...,"Other online resources (e.g., videos, blogs, f...",Stack Overflow;How-to videos;Interactive tutorial,...,No,NaN,Yes,Increase productivity;Greater efficiency;Impro...,Don’t trust the output or answers,7.0,Unknown,NaN,NaN,Learning about a codebase;Project planning;Wri...
4,5,I am a developer by profession,18-24 years old,"Hybrid (some remote, some in-person)","Student, full-time",Norway,NaN,"Secondary school (e.g. American high school, G...","Other online resources (e.g., videos, blogs, f...",Technical documentation;Blogs;Written Tutorial...,...,Yes,Multiple times per day,"No, and I don't plan to",NaN,NaN,7.0,Unknown,NaN,Vim,NaN


## 3. Normalisation de la colonne Country

La colonne `Country` contient des libellés hétérogènes (variantes de noms, formulations anciennes, abréviations...).  
L’objectif est de :

- détecter les valeurs non reconnues comme pays valides ;
- corriger les principaux cas via des remplacements ciblés ;
- marquer certaines entrées comme manquantes lorsqu’il est impossible de les normaliser.

In [2]:
df["Country"].value_counts()

Country
United States of America                                10809
Germany                                                  4797
India                                                    3890
United Kingdom of Great Britain and Northern Ireland     3145
Ukraine                                                  2654
                                                        ...  
Central African Republic                                    1
Equatorial Guinea                                           1
Niger                                                       1
Guinea                                                      1
Solomon Islands                                             1
Name: count, Length: 185, dtype: int64

### 3.1. Détection des pays non valides

In [3]:
# Cette fonction vérifie si le nom de pays fourni correspond à un pays valide selon la norme ISO (via la librairie pycountry).
# Si le pays est reconnu, elle retourne True ; sinon, elle retourne False.
# Cela permet ensuite d’identifier les entrées erronées ou incohérentes dans la colonne "Country".
def country_valid(country_name):
    import pycountry
    try:
        pycountry.countries.lookup(country_name)
        return True
    except LookupError:
        return False

on insére une colonne qui indique si le nom de pays est valide

In [4]:
df["Country_valide"]=df["Country"].apply(country_valid)

In [5]:
# Affichage des Country non valide
print(df[(df["Country_valide"]==False) & (df["Country"].notna())]["Country"].value_counts())

Country
Turkey                                  545
Iran, Islamic Republic of...            410
Hong Kong (S.A.R.)                      145
Venezuela, Bolivarian Republic of...     69
Republic of Korea                        57
Nomadic                                  43
Kosovo                                   18
Palestine                                15
Congo, Republic of the...                 8
Cape Verde                                5
Swaziland                                 4
Libyan Arab Jamahiriya                    4
Democratic Republic of the Congo          3
Micronesia, Federated States of...        1
Name: count, dtype: int64


### 3.2 Corrections manuelles des libellés

Les principaux libellés incohérents ou ambigus sont harmonisés manuellement :


In [6]:
df["Country"]=df["Country"].replace("Turkey","Türkiye")
df["Country"]=df["Country"].replace("Iran, Islamic Republic of...","Iran")
df["Country"]=df["Country"].replace(["Kosovo","Republic of Kosovo"],"Republic of Serbia")
df["Country"]=df["Country"].replace(["Venezuela, Bolivarian Republic","Venezuela","Venezuela, Bolivarian Republic of...","Vénézuéla"],"Venezuela")
df["Country"]=df["Country"].replace("Bolivarian Republic of...","Vénézuéla")
df["Country"]=df["Country"].replace(["Republic of Korea","Republic of Korea (South Korea)"],"Democratic People's Republic of Korea")
df["Country"]=df["Country"].replace("Nomadic",pd.NA)
df["Country"]=df["Country"].replace(["Palestine","State of Palestine"],"the State of Palestine")
df["Country"]=df["Country"].replace("Hong Kong (S.A.R.)","Hong Kong Special Administrative Region of China")
df["Country"]=df["Country"].replace("Democratic Republic of the Congo","")
df["Country"]=df["Country"].replace("Swaziland","Eswatini")
df["Country"]=df["Country"].replace(["Congo","Congo, Republic of the..."],"Republic of the Congo")
df["Country"]=df["Country"].replace("Republic of the...",pd.NA)
df["Country"]=df["Country"].replace("Libyan Arab Jamahiriya","libya")
df["Country"]=df["Country"].replace("Cape Verde","cabo Verde")
df["Country"]=df["Country"].replace(["Micronesia","Micronesia, Federated States of..."],"Federated States of Micronesia")
df["Country"]=df["Country"].replace("Federated States of...",pd.NA)
df["Country"]=df["Country"].replace("",pd.NA)

### 3.3. Vérification après normalisation

In [7]:
df["Country_valide"]=df["Country"].apply(country_valid)
df[(df["Country_valide"]==False) & (df["Country"].notna())]["Country"].value_counts()

Series([], Name: count, dtype: int64)

## 4. Feature engineering : niveau d’expérience (YearsCodePro → ExperienceLevel)

La colonne `YearsCodePro` contient des années d’expérience professionnelle en numérique.  
Pour faciliter l’analyse et la communication des résultats, je crée une variable catégorielle `ExperienceLevel` :

- **débutant** : moins de 5 ans d’expérience ;
- **junior** : entre 5 et 10 ans ;
- **sénior** : 10 ans et plus.


In [8]:
labels=["debutant","junior","sénior"]
bins=[df["YearsCodePro"].min(),5,10,df["YearsCodePro"].max()]
df["ExperienceLevel"]=pd.cut(df["YearsCodePro"],bins=bins,labels=labels,right=False)
df[["YearsCodePro","ExperienceLevel"]]

,YearsCodePro,ExperienceLevel
0,8.0,junior
1,17.0,sénior
2,27.0,sénior
3,8.0,junior
4,8.0,junior
...,...,...
61196,8.0,junior
61197,24.0,sénior
61198,3.0,debutant
61199,5.0,junior


## 5. Normalisation de la colonne Employment

In [9]:
#Extraction des valeurs unique de la colonne Emplyment
dt=pd.DataFrame({"Employment":df["Employment"].unique()})
dt

,Employment
0,"Employed, full-time"
1,"Student, full-time"
2,"Student, full-time;Not employed, but looking f..."
3,"Independent contractor, freelancer, or self-em..."
4,"Not employed, and not looking for work"
...,...
105,"Not employed, but looking for work;Independent..."
106,"Student, full-time;Retired"
107,"Employed, full-time;Not employed, but looking ..."
108,"Not employed, and not looking for work;Student..."



La colonne `Employment` peut contenir plusieurs informations combinées (statut, type d’emploi, situation de recherche, etc.) dans une seule chaîne de caractères.

Objectif : transformer cette colonne en variables binaires plus lisibles :

- `Employed` : actuellement en emploi ;
- `Student` : étudiant ;
- `full_time` : travaille à temps plein ;
- `Independent` : indépendant / freelance ;
- `looking_for_work` : en recherche active d’emploi ;
- `Retired` : retraité.

### 5.1. Exploration des modalités

In [10]:
# Examinant tous les option possible dans la colonne Emplyment
dt["Employment"]=dt["Employment"].str.split(";")
dt=pd.DataFrame(dt.explode(column="Employment")["Employment"].unique()).sort_values(0)
#dt.reset_index(drop=True,inplace=True)
dt

,0
0,"Employed, full-time"
6,"Employed, part-time"
7,I prefer not to say
3,"Independent contractor, freelancer, or self-em..."
4,"Not employed, and not looking for work"
2,"Not employed, but looking for work"
8,Retired
1,"Student, full-time"
5,"Student, part-time"


On represente :
* Employed et Not Employed -> colonne Employed (1 ou 0)
* Student -> colonne Student (1 ou 0)
* Independent  -> colonne Independent  (1 ou 0)
* Retired  -> colonne Retired  (1 ou 0)
* full-time ou part-time -> colonne full-time (1 ou 0)
* Looking for work or no -> colonne Looking_for_work

### 5.2. Création des variables indicatrices

In [11]:
df["Employed"] = df["Employment"].str.contains("Employed").astype(int) # sensible a la casse, ne prends pas emlpyment du "Not emplyment"
df["Student"] = df["Employment"].str.contains("Student").astype(int)
df["full_time"] = df["Employment"].str.contains("full-time").astype(int)
df["Independent"] = df["Employment"].str.contains("Independent contractor, freelancer, or self-employed").astype(int)
df["looking_for_work"] = df["Employment"].str.contains("Not employed, but looking for work").astype(int)
df["Retired"] = df["Employment"].str.contains("Retired").astype(int)

### 5.3. Vérification

In [12]:
df[["Employment","Employed","Student","full_time","Independent","looking_for_work","Retired"]]

,Employment,Employed,Student,full_time,Independent,looking_for_work,Retired
0,"Employed, full-time",1,0,1,0,0,0
1,"Employed, full-time",1,0,1,0,0,0
2,"Employed, full-time",1,0,1,0,0,0
3,"Student, full-time",0,1,1,0,0,0
4,"Student, full-time",0,1,1,0,0,0
...,...,...,...,...,...,...,...
61196,"Not employed, but looking for work;Employed, p...",1,0,0,0,1,0
61197,"Employed, full-time",1,0,1,0,0,0
61198,"Employed, full-time",1,0,1,0,0,0
61199,"Employed, full-time",1,0,1,0,0,0


### 5.4. Suppression de la colonne d’origine

In [13]:
# suppression de la colonne Emplyement 
df=df.drop(columns="Employment")

## 6. Export du dataset amélioré

Le dataset normalisé est sauvegardé sous forme de nouvelle version :

- **`survey-data_V3_correct.csv`** → base de travail pour les notebooks d’EDA et les analyses métiers.


In [14]:
output_path = "../Data/Survey/processed/survey-data_V3_final.csv"
df.to_csv(output_path, index=False)

## 7. Résumé

Dans ce notebook, j’ai :

- normalisé la colonne `Country` en corrigeant et harmonisant les libellés de pays ;
- créé une variable `ExperienceLevel` à partir de `YearsCodePro` pour faciliter l’analyse des profils ;
- décomposé `Employment` en plusieurs indicateurs binaires (`Employed`, `Student`, `full_time`, `Independent`, `looking_for_work`, `Retired`) ;
- supprimé la colonne `Employment` devenue redondante ;
- exporté un dataset `V3` prêt pour l’analyse exploratoire détaillée.

Ce dataset servira de base aux notebooks d’EDA (profils développeurs, stack technologique, salaires, etc.) et à la construction des visualisations finales pour le rapport.
